# Task 2 & 3 – Recommendation Models and Evaluation
**INFO345 – RateBeer dataset**

We build and evaluate the collaborative filtering recommenders from the course (Lecture 3–5, Lab 2 and Lab 3) on our RateBeer data, using **Scikit Surprise** and **5-fold cross validation**. Models are compared on rating prediction accuracy with **RMSE** and **MAE**.

| Model in the table | Surprise class | Type |
|---|---|---|
| SVD | `SVD` | Matrix factorization |
| SVD++ | `SVDpp` | Matrix factorization (+ implicit feedback) |
| KNN-Basic | `KNNBasic` | Neighbourhood-based CF |
| KNN-Baseline | `KNNBaseline` | Neighbourhood-based CF with baseline ratings |
| Baseline | `BaselineOnly` | Global mean + user bias + item bias |
| Random | `NormalPredictor` | Random rating from the rating distribution |

Requires `01_data_analysis.ipynb` to have been run first (it writes `filtered.parquet`).

## Loading the data

In [1]:
import pandas as pd
import numpy as np

DATA = ".."   # the 'RateBeer Dataset' folder

# Ratings after the filtering in Task 1 (2012–2017, users >= 10 ratings, beers >= 5 ratings)
ratings = pd.read_parquet(f"{DATA}/filtered.parquet")
ratings = ratings[['user_id', 'beer_id', 'rating']]
ratings.head()

,user_id,beer_id,rating
0,442761,105273,1.9
1,288889,105273,1.6
2,250510,105273,1.5
3,122778,105273,1.9
4,227834,105273,1.9


SVD++ is very slow for users with thousands of ratings (our most active user has over 20,000). To make all models trainable on a laptop, we keep **at most 100 randomly chosen ratings per user**. All users are kept, and very active users no longer dominate the data.

In [2]:
print("Ratings per user before sampling:")
print(ratings.groupby('user_id').size().describe().round(1))

# keep at most 100 random ratings per user (fixed random_state for reproducibility)
ratings = ratings.sample(frac=1, random_state=42).groupby('user_id').head(100)

num_users = ratings['user_id'].nunique()
num_beers = ratings['beer_id'].nunique()
num_ratings = len(ratings)
density = num_ratings / (num_users * num_beers)

print("\nNumber of users:", num_users)
print("Number of beers rated:", num_beers)
print("Number of ratings:", num_ratings)
print("Data density: {:.4%}".format(density))
print("Data sparsity: {:.4%}".format(1 - density))

Ratings per user before sampling:
count    10368.0
mean       363.1
std       1022.8
min         10.0
25%         18.0
50%         49.0
75%        243.0
max      20670.0
dtype: float64



Number of users: 10368
Number of beers rated: 88828
Number of ratings: 592205
Data density: 0.0643%
Data sparsity: 99.9357%


## Model based approaches with Surprise

In [3]:
#Import the required classes and methods from the surprise library
from surprise import SVD, SVDpp, KNNBasic, KNNBaseline, BaselineOnly, NormalPredictor
from surprise import Reader, Dataset
from surprise.model_selection import cross_validate, KFold
from tabulate import tabulate
import time
import datetime

#Define a Reader object
#The Reader object helps in parsing the dataframe containing ratings
#RateBeer ratings go from 0.5 to 5
reader = Reader(rating_scale=(0.5, 5))

#Create the dataset to be used for building the filter
data = Dataset.load_from_df(ratings[['user_id', 'beer_id', 'rating']], reader)

### K-fold cross validation
We use 5-fold cross validation, which randomly splits the data into 5 equal parts. Each part is used once as test set while the other 4 parts are used for training, and the evaluation scores are averaged over the 5 folds.

In [4]:
#let's make the random a bit less random
np.random.seed(0)
# split data into 5 folds
kf = KFold(n_splits=5, random_state=0)

In [5]:
#The recommenders and their titles
recommenders = [SVD(), SVDpp(), KNNBasic(), KNNBaseline(), BaselineOnly(), NormalPredictor()]
titles = ['SVD', 'SVD++', 'KNN-Basic', 'KNN-Baseline', 'Baseline', 'Random']

In [6]:
table = []
for algo, title in zip(recommenders, titles):
    start = time.time()

    # cross validate on 5 folds on two evaluation metrics
    result = cross_validate(algo, data, ['rmse', 'mae'], kf)

    cv_time = str(datetime.timedelta(seconds=int(time.time() - start)))
    mean_rmse = '{:.3f}'.format(np.mean(result['test_rmse']))
    mean_mae = '{:.3f}'.format(np.mean(result['test_mae']))

    table.append([title, mean_rmse, mean_mae, cv_time])
    print(title, "done in", cv_time)

SVD done in 0:01:44


SVD++ done in 0:07:04


Computing the msd similarity matrix...
Done computing similarity matrix.


Computing the msd similarity matrix...
Done computing similarity matrix.


Computing the msd similarity matrix...
Done computing similarity matrix.


...

Estimating biases using als...


Estimating biases using als...


Estimating biases using als...


Baseline done in 0:00:51


Random done in 0:00:28


### Results

In [7]:
header = ['Recommenders', 'Pred Accuracy [RMSE]', 'Pred Accuracy [MAE]']
print(tabulate([row[:3] for row in table], header, tablefmt="pipe"))

| Recommenders   |   Pred Accuracy [RMSE] |   Pred Accuracy [MAE] |
|:---------------|-----------------------:|----------------------:|
| SVD            |                  0.533 |                 0.392 |
| SVD++          |                  0.52  |                 0.381 |
| KNN-Basic      |                  0.574 |                 0.419 |
| KNN-Baseline   |                  0.533 |                 0.39  |
| Baseline       |                  0.532 |                 0.39  |
| Random         |                  1.001 |                 0.79  |


In [8]:
# The same table including the time used for the 5-fold cross validation
print(tabulate(table, header + ['Time'], tablefmt="pipe"))

| Recommenders   |   Pred Accuracy [RMSE] |   Pred Accuracy [MAE] | Time    |
|:---------------|-----------------------:|----------------------:|:--------|
| SVD            |                  0.533 |                 0.392 | 0:01:44 |
| SVD++          |                  0.52  |                 0.381 | 0:07:04 |
| KNN-Basic      |                  0.574 |                 0.419 | 0:02:19 |
| KNN-Baseline   |                  0.533 |                 0.39  | 0:02:33 |
| Baseline       |                  0.532 |                 0.39  | 0:00:51 |
| Random         |                  1.001 |                 0.79  | 0:00:28 |


#### Which recommendation method performs best according to each evaluation metric?
*(Discuss the results here.)*